# ATC Multi-Agent GRPO Training
**Works on Colab T4 and Kaggle T4/P100.**
Runtime → GPU. Run cells top to bottom.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/GTsingh600/ats.git"
BRANCH     = "yashh"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
EPISODES   = 100
LORA_RANK  = 16
SEED       = 42

import os
_kaggle    = os.path.exists("/kaggle/working")
REPO_DIR   = "/kaggle/working/ATC"      if _kaggle else "/content/ATC"
OUTPUT_DIR = "/kaggle/working/atc-grpo" if _kaggle else "/content/atc-grpo"

os.environ["WANDB_MODE"]            = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"   # single GPU only

print("Platform  :", "Kaggle" if _kaggle else "Colab")
print("Output    :", OUTPUT_DIR)
print("Episodes  :", EPISODES)

In [ ]:
# ── GPU CHECK ─────────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU — enable GPU runtime first"
cc = torch.cuda.get_device_capability()
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CC   : {cc[0]}.{cc[1]}  ({'bf16 AMP OK' if cc[0]>=8 else 'fp16 mode (T4/P100)'})")

In [ ]:
# ── INSTALL ───────────────────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(args),
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f"pip failed: {args[0]}")

print("Installing unsloth...")
pip("unsloth")

print("Installing training stack...")
pip("--force-reinstall", "--no-deps", "trl", "peft", "accelerate")
pip("bitsandbytes", "datasets>=2.20.0", "matplotlib", "numpy", "openai")

# Some TRL versions hard-import llm_blender which breaks GRPOTrainer import.
# Patch it to be optional — writes to disk so subprocesses see it too.
import trl, os as _os
for _f in ("judges.py", "callbacks.py", "grpo_trainer.py"):
    _p = _os.path.join(_os.path.dirname(trl.__file__), "trainer", _f)
    if _os.path.exists(_p):
        _t = open(_p).read()
        if "import llm_blender" in _t and "try:" not in _t[_t.index("import llm_blender")-30:_t.index("import llm_blender")]:
            open(_p, "w").write(_t.replace(
                "import llm_blender",
                "try:\n    import llm_blender\nexcept Exception:\n    llm_blender = None"
            ))
            print(f"  Patched {_f}")

for _k in [k for k in sys.modules if "trl" in k]: del sys.modules[_k]

from unsloth import FastLanguageModel   # noqa
from trl import GRPOTrainer, GRPOConfig # noqa
print("unsloth + trl GRPO : OK")

In [ ]:
# ── CLONE REPO ────────────────────────────────────────────────────────────────
import shutil, subprocess, os, sys
from pathlib import Path

if Path(REPO_DIR).exists(): shutil.rmtree(REPO_DIR)

r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                   capture_output=True, text=True)
if r.returncode != 0:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()
print(f"Cloned  : {REPO_DIR}")
print(f"Commit  : {commit}")
print(f"Output  : {OUTPUT_DIR}")

In [ ]:
# ── SANITY CHECK ──────────────────────────────────────────────────────────────
from training.dataset import build_episode_dataset
samples = build_episode_dataset(n_episodes=2, seed=0)
print(f"Dataset OK — {len(samples)} samples, roles: {sorted({s['agent_role'] for s in samples})}")

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────────────────
import subprocess, sys, os

cmd = [
    sys.executable, "training/train_grpo.py",
    "--model",         MODEL_NAME,
    "--episodes",      str(EPISODES),
    "--lora_rank",     str(LORA_RANK),
    "--n_generations", "4",
    "--seed",          str(SEED),
    "--output_dir",    OUTPUT_DIR,
    "--no_eval",
]
print("Running:", " ".join(cmd))
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
r = subprocess.run(cmd, cwd=REPO_DIR, env=env)
if r.returncode != 0:
    raise RuntimeError(f"Training failed (exit {r.returncode})")

In [ ]:
# ── PLOT CURVES ───────────────────────────────────────────────────────────────
import json
from pathlib import Path
from training.plot_rewards import plot_training_curves

p = Path(OUTPUT_DIR) / "reward_curves.json"
if p.exists():
    plot_training_curves(json.loads(p.read_text()),
                         save_dir=str(Path(OUTPUT_DIR)/"plots"), show=True)
else:
    print("No reward_curves.json — run training first.")

In [ ]:
# ── BEFORE / AFTER EVAL (optional) ────────────────────────────────────────────
import json
from pathlib import Path
from training.eval import evaluate_model, print_comparison
from training.plot_rewards import plot_eval_comparison

TASKS = ["delhi_monsoon_recovery_easy", "bengaluru_irrops_hard"]
N_EP  = 3

base    = evaluate_model("heuristic-baseline", N_EP, TASKS, 99, False, "Baseline")
trained = evaluate_model(OUTPUT_DIR,           N_EP, TASKS, 99, False, "Trained")
print_comparison(base, trained)

out = {"base":    {k:v for k,v in base.items()    if k!="records"},
       "trained": {k:v for k,v in trained.items() if k!="records"}}
eval_p = Path(OUTPUT_DIR) / "eval_results.json"
eval_p.write_text(json.dumps(out, indent=2))
plot_eval_comparison(out, save_dir=str(Path(OUTPUT_DIR)/"plots"), show=True)
print("Saved:", eval_p)

In [ ]:
# ── CHECKPOINT SUMMARY ────────────────────────────────────────────────────────
import json
from pathlib import Path

out = Path(OUTPUT_DIR)
for name in ["adapter_config.json", "adapter_model.safetensors",
             "reward_curves.json", "generation_samples.jsonl"]:
    p = out / name
    status = f"{p.stat().st_size/1024:.0f} KB" if p.exists() else "MISSING"
    print(f"  {name:<35} {status}")

cfg = out / "adapter_config.json"
if cfg.exists():
    c = json.loads(cfg.read_text())
    print(f"\nBase model : {c.get('base_model_name_or_path')}")
    print(f"LoRA rank  : {c.get('r')}")